#### Vorbereitung: Die beiden Graphen verstehen

In [2]:
import networkx as nx
import numpy as np

H = nx.DiGraph([
    ("R","B"),
    ("R","A"),
    ("B","A"),
    ("A","H")
])

K = nx.DiGraph([
    (2,1),
    (2,3),
    (3,2),
    (5,1),
    (5,4),
    (6,2),
    (6,3),
    (6,5)
])

**nx.DiGraph** bedeutet, dass es sich um einen gerichteten Graphen handelt.

Das heißt: Eine Kante von R nach B bedeutet nicht automatisch, dass es auch eine Kante von B nach R gibt.

#### **1. A² für den Knauer-Graphen – wo steht die 2?**
Hier geht es um die Adjazenzmatrix und ihre zweite Potenz.

Die Adjazenzmatrix beschreibt, welche Knoten direkt miteinander verbunden sind.
- 1 bedeutet: Es gibt eine Kante.
- 0 bedeutet: Es gibt keine Kante.
Die 2 in \(A^2\) steht als Hochzahl. Sie bedeutet, dass die Matrix mit sich selbst multipliziert wird:
\[
A^2=A\cdot A
\]

In [5]:
A = nx.to_numpy_array(
    H,
    nodelist=["R", "B", "A", "H"],
    dtype=int
)

print("Adjazenzmatrix A:")
print(A)
print("\n")

A2 = np.linalg.matrix_power(A, 2)

print("A hoch 2:")

print(A2)

Adjazenzmatrix A:
[[0 1 1 0]
 [0 0 1 0]
 [0 0 0 1]
 [0 0 0 0]]


A hoch 2:
[[0 0 1 1]
 [0 0 0 1]
 [0 0 0 0]
 [0 0 0 0]]


**Was bedeutet das Ergebnis?**

Die Werte in \(A^2\) geben an, wie viele Wege der Länge 2 zwischen zwei Knoten existieren.

Beispiele:
- R → B → A: ein Weg der Länge 2.
- R → A → H: ein Weg der Länge 2.
- B → A → H: ein Weg der Länge 2.
Deshalb stehen an diesen Stellen Einsen.

Merke: \(A^2\) beschreibt Wege mit genau zwei Kanten, nicht alle möglichen Wege

#### 2. nx.transitive_closure(H) mit \(W^+\) aus V1 vergleichen

Hier geht es um den transitiven Abschluss eines Graphen.

Ein transitiver Abschluss zeigt alle Verbindungen, die direkt oder über Zwischenknoten erreichbar sind.

Beispiel:
- R → B (direkt)
- B → A (direkt)
- Daraus folgt: R → A (indirekt).
- A → H und R → A ergeben zusätzlich R → H.
Wir können also neue Kanten ergänzen, wenn ein Knoten über einen anderen Knoten erreichbar ist.

In [6]:
TC = nx.transitive_closure(H)

print("Kanten des transitiven Abschlusses:")
print(list(TC.edges()))

Kanten des transitiven Abschlusses:
[('R', 'B'), ('R', 'A'), ('R', 'H'), ('B', 'A'), ('B', 'H'), ('A', 'H')]


Es kommen zwei neue Kanten hinzu:
- R → H
- B → H
Vergleich mit \(W^+\) aus V1

Der transitive Abschluss kann auch mit Matrizen berechnet werden.

Wir verwenden dafür:
\[
W^+=A\lor A^2\lor A^3\lor\ldots
\]

Dabei bedeutet:
- \(A\): direkte Verbindungen.
- \(A^2\): Wege mit genau zwei Kanten.
- \(A^3\): Wege mit genau drei Kanten.
- \(\lor\): logisches ODER.
Für unseren Graphen H gilt:
\[
W^+=
\begin{pmatrix}
0&1&1&1\\
0&0&1&1\\
0&0&0&1\\
0&0&0&0
\end{pmatrix}
\]

Python-Code zum Vergleich:

In [7]:
A = nx.to_numpy_array(
    H,
    nodelist=["R", "B", "A", "H"],
    dtype=int
)

W = A.copy()

for i in range(2, 4):
    W = np.logical_or(
        W,
        np.linalg.matrix_power(A, i) > 0
    )

W_plus = W.astype(int)

print("W+ aus der Matrix:")
print(W_plus)

print("Transitive Closure:")
print(list(TC.edges()))

W+ aus der Matrix:
[[0 1 1 1]
 [0 0 1 1]
 [0 0 0 1]
 [0 0 0 0]]
Transitive Closure:
[('R', 'B'), ('R', 'A'), ('R', 'H'), ('B', 'A'), ('B', 'H'), ('A', 'H')]


#### 3. nx.is_directed_acyclic_graph für H und K; Kreis mit nx.find_cycle(K)
Hier untersuchen wir, ob ein gerichteter Graph einen Kreis besitzt.

Ein gerichteter Kreis bedeutet, dass man durch die Pfeile wieder zum Ausgangsknoten zurückkommt.

In [8]:
print(nx.is_directed_acyclic_graph(H))
print(nx.is_directed_acyclic_graph(K))

True
False


Was bedeutet das?
- True bei H: H ist ein DAG.
- False bei K: K ist kein DAG.
DAG bedeutet Directed Acyclic Graph, also ein gerichteter Graph ohne gerichtete Kreise.

**Warum hat K einen Kreis?**
Betrachten wir diese beiden Kanten:
2---->3,
3---->2

Damit entsteht ein Kreis:

In [9]:
kreis = nx.find_cycle(K)

print(kreis)

[(2, 3), (3, 2)]


**Erklärung**:

NetworkX gibt die Kanten zurück, die den Kreis bilden.
- Erste Kante: 2 → 3
- Zweite Kante: 3 → 2
Damit ist bewiesen, dass K einen gerichteten Kreis besitzt.

**Merke für die Prüfung**:
Funktion	Bedeutung
True	Kein gerichteter Kreis
False	Gerichteter Kreis vorhanden
find_cycle()	Findet einen Kreis

#### **4. list(nx.topological_sort(H))**
Hier geht es um die topologische Sortierung.

Eine topologische Sortierung ordnet die Knoten so an, dass jede gerichtete Kante von einem früheren zu einem späteren Knoten zeigt.

Das funktioniert nur bei gerichteten Graphen ohne Kreise (DAGs).

Unser Graph H ist ein DAG. Deshalb können wir ihn sortieren.

In [10]:
sortierung = list(nx.topological_sort(H))

print(sortierung)

['R', 'B', 'A', 'H']


**Warum ist diese Reihenfolge richtig**

Wir schauen uns die Kanten an:
- R --->B
- R ---<A>
- B ---->A
- A --->

Daraus folgt:
1. R muss vor B kommen.
2. R muss vor A kommen.
3. B muss vor A kommen.
4. A muss vor H kommen.
Deshalb ist die Reihenfolge:
\[
\boxed{R\rightarrow B\rightarrow A\rightarrow H}
\]

Alltagsbeispiel:

Stell dir vor, du musst vier Aufgaben erledigen:
- R: Aufgabe 1
- B: Aufgabe 2
- A: Aufgabe 3
- H: Aufgabe 4
Du kannst Aufgabe 3 erst erledigen, wenn Aufgabe 2 abgeschlossen ist.

Die topologische Sortierung zeigt dir eine gültige Reihenfolge.

Wichtig: Bei einem Graphen mit einem gerichteten Kreis ist eine topologische Sortierung nicht möglich.

#### **5. list(nx.bfs_layers(K, 6))**
Hier verwenden wir die Breitensuche (BFS = Breadth-First Search).

Die Breitensuche untersucht einen Graphen schichtweise.

Wir beginnen beim Knoten 6 und schauen, welche Knoten erreichbar sind.

In [14]:
schichten = list(nx.bfs_layers(K, 6))

print(schichten)

[[6], [2, 3, 5], [1, 4]]


#### 6. Alle Semiwege von B nach H
Jetzt suchen wir alle möglichen einfachen Wege zwischen B und H

In [15]:
nx.all_simple_paths(
    H.to_undirected(),
    "B",
    "H"
)

<generator object _all_simple_paths_graph at 0x0000023BFFC132E0>